RAG Pipeline : Data Ingestion to Vector DB Pipeline

In [1]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\Users\tanuj\OneDrive\Desktop\ChatBotTanuj\tanujChatBotEnv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [44]:
### Read All the pdf files from the directory

def process_all_pdfs(pdf_directory):
    ###Process all pdf files in the directory

    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files in directory {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing file: {pdf_file}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} documents from {pdf_file}")
        except Exception as e:
            print(f"Error loading {pdf_file}: {e}")
    
    return all_documents


In [46]:
all_pdf_documents = process_all_pdfs("../data")
print(f"Total documents loaded: {all_pdf_documents}")

Found 17 PDF files in directory ../data
Processing file: ..\data\pdf\10th class Marksheet.pdf
Loaded 1 documents from ..\data\pdf\10th class Marksheet.pdf
Processing file: ..\data\pdf\12th Class Marksheet.pdf
Loaded 1 documents from ..\data\pdf\12th Class Marksheet.pdf
Processing file: ..\data\pdf\1st Semester doc.pdf
Loaded 1 documents from ..\data\pdf\1st Semester doc.pdf
Processing file: ..\data\pdf\2nd Semester doc.pdf
Loaded 1 documents from ..\data\pdf\2nd Semester doc.pdf
Processing file: ..\data\pdf\3rd Semester doc.pdf
Loaded 1 documents from ..\data\pdf\3rd Semester doc.pdf
Processing file: ..\data\pdf\4th Semester doc.pdf
Loaded 1 documents from ..\data\pdf\4th Semester doc.pdf
Processing file: ..\data\pdf\5th Semester doc .pdf
Loaded 1 documents from ..\data\pdf\5th Semester doc .pdf
Processing file: ..\data\pdf\6th Semester doc.pdf
Loaded 1 documents from ..\data\pdf\6th Semester doc.pdf
Processing file: ..\data\pdf\7th Semester doc.pdf
Loaded 1 documents from ..\data\pdf\

In [47]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    ### Split documents into smaller chunks

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split into {len(split_docs)} chunks.")

    # example of chunk 
    if split_docs:
        print("Example chunk:")
        print("Chunks : ", split_docs[0].page_content[:500])  # Print first 500 characters of the first chunk
        print("Metadata:", split_docs[0].metadata)
    return split_docs


In [48]:
chunks = split_documents(all_pdf_documents)

Split into 41 chunks.
Example chunk:
Chunks :  Roll No.1611557076 
Reg. No. 151557210076 
Sr. No. 162C0011127203
imacha 
pimachal 
Pra 
nbesh Board of 
ol 
Educatir 
bucation 
hal 
Brades 
MATRICULATION EXAMINATION CERTIFICATE CUM DETAIL OF MARKS 
Session: MARCH 2016 
This is to certify that 
TANUJ 
Sex: MALE 
Father's Name Shrl 
PIAR CHANDD 
Mother's Name Smt. 
NISHA KUMARI 
Born on 
14-03-2001
(Fourteenth March Two Thousand and One) 
has passed the Matriculation Examination of this Board from GOVT SR SEC SCHOOL, 
GAARI (School Code. 1557) 
Metadata: {'producer': 'Adobe Scan for Android 21.03.22-regular', 'creator': 'Adobe Scan for Android 21.03.22-regular', 'creationdate': '', 'source': '10th class Marksheet.pdf', 'file_path': '..\\data\\pdf\\10th class Marksheet.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}


Embedding and VectorStore DB

In [49]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from typing import List, Dict, Any, Tuple
import uuid
from sklearn.metrics.pairwise import cosine_similarity


In [50]:
class EmbeddingManager:
    ###Handles document embedding generation using SetenceTransformer

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args: 
            model_name (str): HuggingFace model name for Sentence embeddings

        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts (List[str]): List of text strings to embed
        Returns:
            np.ndarray: Array of embeddings
        """
        if not self.model:
            raise ValueError("Model is not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts......")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
    def add_documents(self, documents: List[Any], embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents (List[Any]): _description_
            embeddings (np.ndarray): _description_
        """
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store....")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadata.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # embedding
            embeddings_list.append(embedding.tolist())
            
            
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
    

## initialize embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model: sentence-transformers/all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


Vector Store

In [51]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector Store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        
        try:
            # create persistant ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description:":"Pdf document embeddings for RAG"}
            )
            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings:np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents (List[Any]): _description_
            embeddings (np.ndarray): _description_
        """
        if len(documents)!= len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store....")
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            #prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # embedding
            embeddings_list.append(embedding.tolist())
            
            
        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                documents=documents_text,
                embeddings=embeddings_list
            )
            print(f"Successfully added {len(documents)} documents to vector store.")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
            
            
vectorstore = VectorStore()
vectorstore
        
            

Vector Store initialized. Collection: pdf_documents
Existing documents in collection: 61


In [52]:
chunks

[Document(metadata={'producer': 'Adobe Scan for Android 21.03.22-regular', 'creator': 'Adobe Scan for Android 21.03.22-regular', 'creationdate': '', 'source': '10th class Marksheet.pdf', 'file_path': '..\\data\\pdf\\10th class Marksheet.pdf', 'total_pages': 1, 'format': 'PDF 1.3', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'file_type': 'pdf'}, page_content="Roll No.1611557076 \nReg. No. 151557210076 \nSr. No. 162C0011127203\nimacha \npimachal \nPra \nnbesh Board of \nol \nEducatir \nbucation \nhal \nBrades \nMATRICULATION EXAMINATION CERTIFICATE CUM DETAIL OF MARKS \nSession: MARCH 2016 \nThis is to certify that \nTANUJ \nSex: MALE \nFather's Name Shrl \nPIAR CHANDD \nMother's Name Smt. \nNISHA KUMARI \nBorn on \n14-03-2001\n(Fourteenth March Two Thousand and One) \nhas passed the Matriculation Examination of this Board from GOVT SR SEC SCHOOL, \nGAARI (School Code. 1557) DISTRICT HAMIRPUR and pl

In [53]:
### Convert the text to embeddings

texts = [doc.page_content for doc in chunks]

## Generate embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## store in vector db
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 41 texts......


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:01<00:00,  1.71it/s]

Generated embeddings with shape: (41, 384)
Adding 41 documents to vector store....
Successfully added 41 documents to vector store.
Total documents in collection: 102


Retriever Pipeline From VectorStore

In [11]:
class RAGRetriever:
    """handles retrieval of relevant documents from vector store for RAG"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever
        
        Args:
            vector_store (VectorStore): Instance of the VectorStore
            embedding_manager (EmbeddingManager): Instance of the EmbeddingManager
            top_k (int): Number of top similar documents to retrieve
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    
    def retrieve(self, query: str, top_k: int=2 , score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve top_k similar documents for the given query
        
        Args:
            query (str): User query string
            top_k (int): Number of top similar documents to retrieve
        Returns:
            List[Dict[str, Any]]: List of retrieved documents with metadata
        """
        print(f"Generating embedding for query: {query}")
        print(f"Top k : {top_k}, Score threshold: {score_threshold}")
        
        #Generate embedding for query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search query in vector store
        try:
            
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            'distance': distance,
                            'rank': i+1
                        })
                        
                print(f"Retrieved {len(retieved_docs)} documents after applying score threshold.")
            else:
                print("No documents retrieved from vector store.") 
                
            return retieved_docs
        
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []
         
         
ragRetriever = RAGRetriever(vectorstore, embedding_manager)

In [12]:
ragRetriever

In [13]:
ragRetriever.retrieve("12th class marksheet")

Generating embedding for query: 12th class marksheet
Top k : 2, Score threshold: 0.0
Generating embeddings for 1 texts......


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.82it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents after applying score threshold.


[{'id': 'doc_0eda4e33_2',
  'document': "Sr. No. 18/C002/032969 \nReg. No. 151557210076 \nRoll No. 1841557112 \np i m a c h a l  \n3 9  \nimachal \n39, \nSENIOR SECONDARY (PLUS TWo) EXAMINATION CERTIFICATE \nal \n3Dradesh Board of School Edr \nDucation \nCUM DETAIL OF MARKS \nSession: MARCH-2018 \nGroup \nSCIENCE \nThis is to certify that \nTANUJ \nSex \nMALE \nFather's Name Shri \nPIAR CHAND \nMother's Name Smt. \nNISHA KUMARI \nhas passed the Senior Secondary (Plus Two) Examination of this Board from GoVT SR SEC SCHOOL \nGAARLI (School Code. 1557) DISTRICT HAMIRPUR and placed in FIRST Division. \nDETAIL OF MARKs \nMarks \nMaximum \nSr. No. Subject/s \nObtained \nMarks \nEnglish \n74 \n100 \nPhysics \nW/P 54/25 \n79 \n100 \nChemistry \nW/P 47/25 \n100 \nMathematics \n89 \n100 \nComputer Science\nW/P 66/25 \n91* \n100 \nGrace Marks for Division Improvement, if any \nTotal FOUR HUNDRED AND FIVE ONLY \n405 \n500 \n1 \nDenates Distinction i.e. 75% Marks or above. \nL \nDharamshala \nDated

 Integration Vectrodb Context pipeline With LLM Output

In [24]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

# initialize grok LLM
llm = ChatGroq(api_key=groq_api_key, model="llama-3.1-8b-instant", temperature=0.1, max_tokens=1024)

# Simmple RAG function : retrieve + generate answer
def rag_answer(query: str, retriever: RAGRetriever, llm: ChatGroq, top_k=3) -> str:
    """Generate answer for the query using RAG approach"""
    
    # Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k, score_threshold=0.1)
    
    if not retrieved_docs:
        return "No relevant documents found to answer the query."
    
    # Prepare context from retrieved documents
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.4f}):\n{doc['document']}" for doc in retrieved_docs])
    
    # Prepare prompt for LLM
    prompt = f"""Use the following context to answer the question. 
                Context : {context}
                Question: {query}
                Answer:
                """
    
 
    # Generate answer using LLM
    answer = llm.invoke(prompt)
    
    return answer.content

In [26]:
answer = rag_answer("What is the 4th semester marksheet details?", ragRetriever, llm, top_k=2)
print("Answer:", answer)

Generating embedding for query: What is the 4th semester marksheet details?
Top k : 2, Score threshold: 0.1
Generating embeddings for 1 texts......


Batches: 100%|██████████| 1/1 [00:00<00:00, 95.54it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents after applying score threshold.
Answer: No relevant documents found to answer the query.


Enhanced RAG Pipeline Features

In [56]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with advanced options
    Returns source, confidence score, and optionally full context.
    """
    
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {"answer": "No relevant documents found to answer the query.", "sources": [], "confidence_scores": 0.0, "context": ""}
    
    # Prepare context and sources
    context = "\n\n".join([doc["document"] for doc in results])
    sources = [{
        'sources': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['document'][:200] + '...'
    } for doc in results]
    confidence = max(doc['similarity_score'] for doc in results)
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext : {context}\nQuestion: {query}\nAnswer:"""
    answer = llm.invoke(prompt).content
    
    
    
    output = {
        "answer": answer,
        "sources": sources,
        "confidence_scores": confidence
    }
    if return_context:
        output["context"] = context
    
    return output

In [69]:
result = rag_advance("Give me the marks scored in 10th class", ragRetriever, llm, top_k=5, min_score=0.1, return_context=True)
print("Answer:", result["answer"])
print("Sources:", result["sources"])
print("Confidence Score:", result["confidence_scores"])
print("Context:", result["context"][:300])

Generating embedding for query: Give me the marks scored in 10th class
Top k : 5, Score threshold: 0.1
Generating embeddings for 1 texts......


Batches: 100%|██████████| 1/1 [00:00<00:00, 96.03it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after applying score threshold.


Answer: The marks scored in 10th class (Matriculation Examination) by Tanuj are:

- English: 87/100
- Mathematics: 99/100
- Hindi: 93/100
- Social Science (His. Civ. Geog.): 88/100
- Science & Technology (Phy. Chem. Life Sci.): 87/100
- Sanskrit: 82/100
- Computer Science (Elective): 97/100

Total marks scored: 633/700
Sources: [{'sources': '12th Class Marksheet.pdf', 'page': 0, 'score': 0.23950034379959106, 'preview': 'Sr. No. 18/C002/032969 \nReg. No. 151557210076 \nRoll No. 1841557112 \np i m a c h a l  \n3 9  \nimachal \n39, \nSENIOR SECONDARY (PLUS TWo) EXAMINATION CERTIFICATE \nal \n3Dradesh Board of School Edr \nDucatio...'}, {'sources': '12th Class Marksheet.pdf', 'page': 0, 'score': 0.23950034379959106, 'preview': 'Sr. No. 18/C002/032969 \nReg. No. 151557210076 \nRoll No. 1841557112 \np i m a c h a l  \n3 9  \nimachal \n39, \nSENIOR SECONDARY (PLUS TWo) EXAMINATION CERTIFICATE \nal \n3Dradesh Board of School Edr \nDucatio...'}, {'sources': '10th class Marksheet.pdf', 'page': 0